# Seminar HCI and BCI in practice
## Session 5 Feature preconditioning and extraction

***Choosing the best features through the use of t-values***

In [ ]:
import numpy as np
import os
import sys
import pickle
%matplotlib qt
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from scipy.stats import zscore

sys.path.append(os.path.join(os.getcwd(), "src"))
from create_subsets import create_subsets
from plot_features import plot_features

main_path = os.getcwd()
data_path = os.path.join(main_path, 'data')
print(f'Now you are located: {main_path}')


In [ ]:
ecog_file = os.path.join(data_path, 'raw/ecogStruct3.pkl')
with open(ecog_file, 'rb') as f:
    ecog = pickle.load(f)

# Take a look into our data again
print("ecog contains")
for key, value in ecog.items():
    print(f"Key:{key}, Type:{type(value)}")

In [ ]:
# Load z-scored data from last session
zScoredData_file = os.path.join(data_path, 'raw/zScoredData.pkl')
with open(zScoredData_file, 'rb') as f:
    dat, nFreq, nChan, nTrials = pickle.load(f)

print(f"z-scored data shape is: {np.array(dat).shape}")
print(f"frequency number: {nFreq}")
print(f"channel number: {nChan}")
print(f"trial number: {nTrials}")

In [ ]:
# Load epoch info
epoch_file = os.path.join(data_path, 'raw/epoch2.pkl')
with open(epoch_file, 'rb') as f:
    epoch = pickle.load(f)

for key, value in epoch.items():
    print(f"Key:{key}, Type:{type(value)}")

## t-Values, to choose the best features (using the z-scored data)

### Subsets

Features for classification are often selected from the training data set. In avoidance of being to specific features are typically determined from a subset of the whole training data.

In [ ]:
# Create subsets (ratio of entire data, i.e. 0.9 means 90 %)
subSet1, subSet2 = create_subsets(dat,epoch,0.9)    # subSet1 will contain 90% of all finger flexion trials
                                                    # subSet2 will contain 90% of all finger extension trials

print(subSet1.shape)
print(subSet2.shape)

---
<h2 style="color: #FF0000; font-weight: bold;">TASK 1 (1 pt):</h2>

Have a short look at the `create_subsets` function to fully understand how the subsets are created. This function can only be used for the finger flexion/extension classes. Why?

In [ ]:
# --- TASK 1: which labels does create_subsets actually look for? ---
labels = np.array(epoch['label'])
values, counts = np.unique(labels, return_counts=True)
print("labels found in epoch2:", dict(zip(values.tolist(), counts.tolist())))
print("20 = flexion, 21 = extension")
print("\ncreate_subsets keeps label == 20 for class 1 and label == 21 for class 2,")
print("so subSet1 has", subSet1.shape[0], "trials and subSet2 has", subSet2.shape[0], "trials")

<h3 style="color: #FF0000; font-weight: bold;">Your Answers or Code demostration: </h3>

**Why can `create_subsets` only be used for the finger flexion/extension classes?**

Because the two labels are **hard-coded** inside the function. It does not look at how many classes there are, it simply searches for two fixed numbers (lines 31 and 32):

```python
class1 = np.where(np.array(labels) == 20)[0]   # flexion
class2 = np.where(np.array(labels) == 21)[0]   # extension
```

20 and 21 are exactly the labels that were given to the flexion and extension epochs back in Session 1 and 2. In our `epoch2` there are only these two: **163 trials with label 20 and 151 with label 21**. The function then takes a random `ratio` part of each and returns exactly **two** subsets, `subSet1` and `subSet2`.

So the function is built around this one specific two-class problem:

- It only knows about the values 20 and 21. Any trial with a different label would just be ignored, and if the data used other label numbers, both subsets would come back empty.
- It always returns **exactly two** subsets. It cannot handle three or more classes, for example the five different gestures from Session 1. For those you would need a version that loops over all the labels that are actually present instead of the two fixed numbers.

In short: it is not a general subset function, it is written specifically for the flexion vs. extension comparison, because those class labels are written directly into the code.

## T-Values:
We will use t-values to compare the subsets. 

<h2 style="color: #FF0000; font-weight: bold;">TASK 2 (2 pt):</h2>

Based on the following formular, calculate the t-values manually:


$$ t = \frac
            {
            \bar{X}_{1} - \bar{X}_2
            }
            {
            \sqrt
                {
                (
                    {\frac
                        {(N_1 - 1) S_1^2 + (N_2 - 1) S_2^2}
                        {N_1 + N_2 - 2}
                    }
                )
                (
                    {\frac{1}{N_1}} + 
                    {\frac{1}{N_2}}
                )
                }
            } $$ 

<h3 style="color: #FF0000; font-weight: bold;">Fill in the missing parts (...) in the code below</h3>


In [ ]:
# Calculate variance and mean for each feature
xbar1 = np.mean(subSet1, axis=0)
xbar2 = np.mean(subSet2, axis=0)

# Calculate sample size for each subset
N1 = subSet1.shape[0]
N2 = subSet2.shape[0]

# Calculate variance (with ddof=1 for sample variance)
s1 = np.var(subSet1, axis=0, ddof=1)
s2 = np.var(subSet2, axis=0, ddof=1)

# Then calculate the t-values using the formular
tVals = (xbar1 - xbar2) / np.sqrt(((N1 - 1) * s1 + (N2 - 1) * s2) / (N1 + N2 - 2) * (1 / N1 + 1 / N2))


In [ ]:
# plot the ABSOLUTE t-values: a strong difference is strong in either direction
plot_features(np.abs(tVals), ecog['selectedChannels'], nFreq)

<h2 style="color: #FF0000; font-weight: bold;">TASK 3 (2 pt):</h2>

First have a look at the `plot_features` function to understand how this plot was created. 
- What information can you get from this plot?
- What do the colors mean?
- Why do we plot the absolute tVals?

Make sure you write your observations down, as we will need them next week for the classification.

You can compare your results to the anatomical image

In [ ]:
# --- TASK 3: which channels and frequencies separate flexion from extension best? ---
absT = np.abs(tVals)
# reshape the same way plot_features does, so it matches the image: (nChan, nFreq)
tMatrix = np.reshape(absT, (nChan, nFreq), order='F')

# rebuild the frequency axis (freqIdx was made in Session 4; here we redo it from centerFrequency)
centerFreq = np.array(ecog['periodogram']['centerFrequency'])
freqBand = np.array(list(range(4, 59)) + list(range(62, 119)) + list(range(122, 179)))
freqIdx = np.unique([np.argmin(np.abs(centerFreq - fb)) for fb in freqBand])
freqHz = centerFreq[freqIdx]

print("feature grid: %d channels x %d frequencies (%.1f - %.1f Hz)"
      % (nChan, nFreq, freqHz.min(), freqHz.max()))
print("overall largest |t| = %.2f\n" % absT.max())

# strongest channel for the classification, and at which frequency
best_per_chan = tMatrix.max(axis=1)
print("top channels by their best |t|:")
for k in np.argsort(-best_per_chan)[:5]:
    ch = ecog['selectedChannels'][k]
    f_at = freqHz[np.argmax(tMatrix[k])]
    print("   channel %2d : |t| = %.2f  at %3.0f Hz" % (ch, best_per_chan[k], f_at))

<h3 style="color: #FF0000; font-weight: bold;">Your Answers: </h3>

**How the plot is made.** `plot_features` takes the t-value for every feature and reshapes it back into a **channels x frequencies** grid (`np.reshape(..., order='F')`, the same Fortran order as before). It then puts these values into a full 40-channel image, where the rejected channels stay at 0, and shows it with `imshow`. So the y-axis is the electrodes, the x-axis is the frequencies, and each pixel is one feature.

**What information can I get from this plot?**

It shows **which features separate finger flexion from extension best**, so which channel at which frequency is useful for the classification next week. A bright pixel means that this channel, at this frequency, has very different power in the two classes, so it is a good feature. A dark pixel means the two classes look almost the same there, so that feature does not help.

In our data the strongest features are not spread out evenly, they sit on a few channels:

- **channel 17** is the best by far, |t| = 7.15 at about 157 Hz,
- then **channel 23** (|t| = 5.64 at ~101 Hz), **channel 39** and **channel 30**.

So only a small number of electrodes carry most of the information, and they are strong mainly in the **higher frequencies / high-gamma range**, which is where movement related activity is expected. When I compare this with the anatomical image, these should be the electrodes lying over the hand area of the motor cortex. The rejected channels are just empty rows, because they were set to 0.

**What do the colors mean?**

The color is the **size of the t-value** of that feature. Because I plot the absolute t-values, bright means a large \|t\|, so a big and reliable difference between flexion and extension, and dark means \|t\| close to 0, so almost no difference. The colorbar on the side gives the scale.

**Why do we plot the absolute t-values?**

The sign of the t-value only tells us **which** class has the higher power (flexion or extension), but for choosing good features we do not care about the direction, only about **how strongly** a feature separates the two classes. A feature with t = -7 is just as useful as one with t = +7, they both separate the classes very well, only in opposite directions.

If I plotted the raw t-values, the strong negative features would show up as dark, exactly like the useless features near 0, and I would miss them. Taking the absolute value puts "strongly flexion" and "strongly extension" on the same bright end of the scale, so the plot shows the **usefulness** of each feature regardless of its direction. That is what matters for feature selection.

**For next week:** the most useful features are on channels 17, 23, 30 and 39, mostly in the higher frequency band, and these are the ones worth keeping for the classifier.

In [ ]:
# Load and display the anatomy image
img_path = os.path.join(main_path, 'docs/figs/GP33_anatomy_40_electrodes.png')
anatomy_img = mpimg.imread(img_path)

plt.figure(figsize=(30, 26))
plt.imshow(anatomy_img)
plt.axis('off')
plt.show()